# Day 3 — 비동기 처리와 에러 핸들링

모델 배포 개론 · DAY 03
범위: §1 ~ §6

> **오늘의 한 문장**
> 기다림은 겹치고, 무거운 추론은 옆방(스레드풀)으로 떼어내고,
> 예외는 한곳에서 잡고, 모든 일은 로그로 남깁니다.
> Day 2의 친절한 서버가, 여러 명이 동시에 불러도 죽지 않는 서버가 됩니다.

## 학습 목표 — 네 가지면 충분합니다

| # | 목표 | 핵심 |
|---|------|------|
| 01 | **왜 멈추는지 안다** | 이벤트 루프 = 일꾼 한 명 구조 |
| 02 | **기다림을 겹친다** | 동기 9초 vs 비동기 3초 |
| 03 | **무거운 계산은 떼어낸다** | `run_in_executor`로 옆방 위임 |
| 04 | **터져도 죽지 않게 한다** | 전역 예외 처리 + 로깅 |

> 💡 이 노트북은 Day 1·2에서 만든 `app/model_utils.py`, `app/schemas.py`,
> `models/mnist_state_dict.pth`를 그대로 재사용합니다.
> 커널은 반드시 **Model Serving**(.venv)으로 전환하세요.

## 0. 준비 — 공통 헬퍼와 환경 확인

주피터 노트북 안에서 FastAPI 서버를 띄우기 위한 `serve_in_thread` / `stop_server`
헬퍼를 정의합니다. (터미널에서는 `uvicorn app.main:app` 으로 실행하면 됩니다.)

In [1]:
import sys, os
print(f"Python 경로: {sys.executable}")
print(f"현재 위치: {os.getcwd()}")
# 출력에 .venv가 있어야 Model Serving 커널입니다.

Python 경로: C:\Users\TOP\model-serving-course\.venv\Scripts\python.exe
현재 위치: C:\Users\TOP\model-serving-course


In [2]:
# === 노트북용 서버 실행 헬퍼 (그대로 실행만 하세요) ===
import asyncio, threading, time, socket
import uvicorn

_SERVERS = {}

def _port_open(host, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    """실행 중인 서버를 멈춥니다."""
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host="127.0.0.1", port=8000, log_level="warning"):
    """백그라운드 스레드에서 uvicorn 서버를 띄웁니다.
    app: FastAPI 객체 또는 'app.main:app' 같은 import 경로.
    같은 포트에 서버가 있으면 먼저 멈추고 새로 띄웁니다."""
    stop_server(port)
    if isinstance(app, str):
        sys.modules.pop(app.split(":")[0], None)  # 파일 수정분 반영
    for _ in range(50):
        if not _port_open(host, port):
            break
        time.sleep(0.1)
    config = uvicorn.Config(app, host=host, port=port,
                            log_level=log_level, loop="asyncio")
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def _run():
        if sys.platform == "win32":
            loop = asyncio.SelectorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    for _ in range(40):
        if _port_open(host, port):
            print(f"서버 실행됨: http://{host}:{port}")
            return server
        time.sleep(0.25)
    print("서버가 시작되지 않았습니다. 위 로그를 확인하세요.")
    return server

print("서버 도우미 준비 완료 (serve_in_thread, stop_server)")

서버 도우미 준비 완료 (serve_in_thread, stop_server)


## §1. 왜 서버가 멈추는가?

### §1.1 한 명에겐 완벽했던 서버, 셋이 부르면 9초?

Day 2의 API는 혼자 쓸 땐 흠잡을 데가 없었습니다.
그런데 동시에 세 명이 부르면 — 마지막 사람은 **9초**를 기다립니다.

```
추론 1건 = 3초
A·B·C가 동시 도착 →  A: 3초,  B: 6초,  C: 9초
```

서버가 느려서가 아닙니다. CPU는 놀지 않습니다. **문제는 구조**입니다.
요청을 처리하는 '일꾼'이 사실상 한 명이라, 앞사람 일이 끝나야 뒷사람을 시작합니다.

### §1.2~1.3 동기는 줄을 세우고, 비동기는 틈을 쓴다

```
동기  : A → B → C  (줄줄이)        →  마지막 9초
비동기: A·B·C 겹쳐서 흐름           →  셋 다 3초
```

☕ **카페 비유**: 바리스타 한 명이 A의 머신이 도는 '틈'에 B 주문을 받고
C 블렌더를 켭니다 — 같은 한 명이 9분 일을 3분에 끝냅니다.

> 바뀐 건 '전략'뿐. 서버를 늘리지도, CPU를 키우지도 않았습니다.

### §1.4 작업의 두 종류 — 비동기의 재료는 '기다림'

| 종류 | 예시 | 비동기 효과 |
|------|------|------------|
| **I/O-bound** (입출력 중심) | DB, 외부 API, 파일 | ✅ 기다림이 있어 효과 큼 |
| **CPU-bound** (계산 중심) | **모델 추론** | ❌ 기다림이 없어 효과 없음 |

> ⚠️ 모델 추론은 CPU-bound입니다. 엔드포인트를 `async`로 바꿔도
> 추론은 1초도 빨라지지 않습니다. 오히려 잘못 쓰면 서버 전체가 멈춥니다.

### §1.5 이벤트 루프 — FastAPI의 일꾼은 사실상 '한 명'

이벤트 루프는 FastAPI 안에서 할 일 목록을 끝없이 돌며 처리하는 **단 하나의 일꾼**입니다.
한 작업이 '기다림'에 들어가면 잽싸게 다음 작업으로 갈아탑니다.
단, 한 명이라 누군가 양보 없이 붙들면 **전부 멈춥니다.**

#### 🔑 오늘의 함정 — 같은 추론, 어디서 돌리느냐가 갈림길

```python
# ① 일반 def — FastAPI가 자동으로 옆방(스레드)에 보낸다 → 안전
@app.post("/predict")
def predict(req):
    return model(tensor)      # 옆방에서 3초, 루프는 자유 ✅

# ② async def 안에서 무거운 계산 — 오늘의 함정!
@app.post("/predict")
async def predict(req):
    return model(tensor)      # 루프 '위'에서 3초 → /health도 막힘 ❌
```

> 핵심은 `async`를 붙였는지가 아니라, **무거운 계산이 루프를 붙드는지**가 본질입니다.

### ✅ §1 체크포인트
1. 추론 3초짜리 서버에 3명이 동시에 오면 마지막 사람이 9초 걸리는 이유는?
2. I/O-bound와 CPU-bound의 차이는? 모델 추론은 어느 쪽입니까?
3. 이벤트 루프가 '한 명'인데도 여러 요청을 빠르게 처리하는 비결은?
4. `async def` 안에서 무거운 추론을 직접 돌리면 왜 위험합니까?

## §3. 문제 재현 — 일부러 멈춰 본다

진짜 모델 대신 **'3초 기다리기'로 추론 시간을 정확히 통제**합니다.
함정 버전(`async def` 위에서 blocking)과 옆방 버전을 비교합니다.

> `time.sleep(3)`은 CPU-bound 추론처럼 **이벤트 루프를 붙드는** 동작을 흉내 냅니다.
> (`asyncio.sleep`이 아니라 일반 `time.sleep`인 점이 핵심)

In [3]:
%%writefile app/main_blocking.py
"""[함정 버전] async def 안에서 blocking 추론을 직접 실행 — 루프가 멈춘다"""
import time
from fastapi import FastAPI

app = FastAPI(title="Blocking Server (함정)")

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/predict")
async def predict():
    time.sleep(3)          # ❌ 루프 '위'에서 3초 — 그동안 전부 멈춤
    return {"result": "done"}

Overwriting app/main_blocking.py


In [4]:
%%writefile app/main_threaded.py
"""[옆방 버전] 일반 def — FastAPI가 자동으로 스레드풀에 보낸다"""
import time
from fastapi import FastAPI

app = FastAPI(title="Threaded Server (옆방)")

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/predict")
def predict():             # ✅ def → 옆방(스레드)에서 3초, 루프는 자유
    time.sleep(3)
    return {"result": "done"}

Overwriting app/main_threaded.py


In [5]:
# 동시 요청을 던지고 각자 걸린 시간을 재는 헬퍼
import requests, threading, time

def fire_concurrent(n=3, url="http://127.0.0.1:8000/predict"):
    results = {}
    def worker(idx):
        t0 = time.time()
        requests.post(url)
        results[idx] = time.time() - t0
    threads = [threading.Thread(target=worker, args=(i,)) for i in range(n)]
    t_start = time.time()
    for t in threads: t.start()
    for t in threads: t.join()
    total = time.time() - t_start
    for i in sorted(results):
        print(f"  요청 #{i}: {results[i]:.2f}초")
    print(f"  전체 소요: {total:.2f}초")
    return results

def health_latency(url="http://127.0.0.1:8000/health"):
    t0 = time.time()
    r = requests.get(url)
    dt = time.time() - t0
    print(f"  /health 응답: {dt:.2f}초 (status={r.status_code})")
    return dt

### §3 실험 A — 함정 버전 (blocking)

3개 요청을 동시에 던지면 **3·6·9초**로 순차 실행됩니다.

In [6]:
serve_in_thread("app.main_blocking:app", port=8000)
time.sleep(1)
print("[함정 버전] 동시 요청 3개:")
fire_concurrent(3)

서버 실행됨: http://127.0.0.1:8000
[함정 버전] 동시 요청 3개:
  요청 #0: 9.01초
  요청 #1: 3.01초
  요청 #2: 6.01초
  전체 소요: 9.01초


{1: 3.008988380432129, 2: 6.010310888290405, 0: 9.012527704238892}

**추론 중 헬스체크** — 추론이 도는 동안 `/health`도 같이 막힙니다.

In [7]:
import threading, time
# 백그라운드로 추론 1건을 던져 두고, 0.5초 뒤 헬스체크
threading.Thread(target=lambda: requests.post("http://127.0.0.1:8000/predict")).start()
time.sleep(0.5)
print("[함정 버전] 추론 도는 중 헬스체크:")
health_latency()   # 2.5초 안팎 — 맥박이 멈춘다

[함정 버전] 추론 도는 중 헬스체크:
  /health 응답: 2.50초 (status=200)


2.503899574279785

### §3 실험 B — 옆방 버전 (일반 def)

같은 3초인데 **셋 다 3초 안팎**, 헬스체크는 **즉시(0.0초)** 응답합니다.

In [8]:
serve_in_thread("app.main_threaded:app", port=8000)
time.sleep(1)
print("[옆방 버전] 동시 요청 3개:")
fire_concurrent(3)

서버 실행됨: http://127.0.0.1:8000
[옆방 버전] 동시 요청 3개:
  요청 #0: 3.01초
  요청 #1: 3.01초
  요청 #2: 3.01초
  전체 소요: 3.01초


{0: 3.012040615081787, 1: 3.0123062133789062, 2: 3.0123627185821533}

In [9]:
threading.Thread(target=lambda: requests.post("http://127.0.0.1:8000/predict")).start()
time.sleep(0.5)
print("[옆방 버전] 추론 도는 중 헬스체크:")
health_latency()   # 0.0초 — 맥박 즉시

[옆방 버전] 추론 도는 중 헬스체크:
  /health 응답: 0.00초 (status=200)


0.002404928207397461

### §3.4 가장 위험한 증상 — 맥박이 멈춘 서버는 퇴출된다

헬스체크는 서버의 **맥박**입니다. 로드밸런서는 몇 초마다 `/health`를 찔러 생사를 확인합니다.

```
추론이 루프 붙듦 → 헬스체크 지연 → "죽었다" 판단 → 트래픽에서 제외
→ 남은 서버에 부하 몰림 → 도미노 → 전면 장애
```

> 결론: **일꾼(루프)은 늘 비워 둬야 합니다.** 무거운 일은 옆방으로.

## §4. 해법 — `run_in_executor`

### §4.0 스레드풀 = '옆방 일꾼들'

- **스레드** = 일손 하나
- **스레드풀** = 일손을 미리 N명 모아둔 대기조
- FastAPI 기본 풀 = **40명** (직접 조절 불가, 모든 `def`가 공유)
- → 그래서 추론 전용 풀을 직접 만든다!

### §4 오늘의 해법 — 코드는 이 한 장이면 충분

Day 2 서버에서 **추론 호출 한 줄만** 바꿉니다. 검증·전처리·응답은 그대로입니다.

In [10]:
%%writefile app/main_executor.py
"""[전용 풀 버전] run_in_executor로 추론만 옆방으로 위임"""
import asyncio, time
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI

app = FastAPI(title="Executor Server (전용 풀)")

# 추론 전용 일꾼 4명을 미리 만들어 둔다 (= 스레드풀)
pool = ThreadPoolExecutor(max_workers=4)

def heavy_inference():
    time.sleep(3)          # 무거운 추론 흉내
    return {"result": "done"}

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/predict")
async def predict():
    loop = asyncio.get_event_loop()
    # 무거운 추론만 옆방(전용 풀)으로 위임, await가 결과를 회수
    result = await loop.run_in_executor(pool, heavy_inference)
    return result

Overwriting app/main_executor.py


In [11]:
serve_in_thread("app.main_executor:app", port=8000)
time.sleep(1)
print("[전용 풀 버전] 동시 요청 3개:")
fire_concurrent(3)

서버 실행됨: http://127.0.0.1:8000
[전용 풀 버전] 동시 요청 3개:
  요청 #0: 3.01초
  요청 #1: 3.01초
  요청 #2: 3.01초
  전체 소요: 3.01초


{1: 3.0054848194122314, 2: 3.005964994430542, 0: 3.0073070526123047}

In [12]:
threading.Thread(target=lambda: requests.post("http://127.0.0.1:8000/predict")).start()
time.sleep(0.5)
print("[전용 풀 버전] 추론 도는 중 헬스체크:")
health_latency()   # 0.0초

[전용 풀 버전] 추론 도는 중 헬스체크:
  /health 응답: 0.00초 (status=200)


0.0026934146881103516

### §4.5 스레드풀 크기 가이드

| 환경 | 권장 크기 | 이유 |
|------|----------|------|
| **CPU 추론** | ≈ 코어 수 (`os.cpu_count()`) | 손 바꿔쥐는 낭비 방지 |
| **GPU 추론** | 1~2 | GPU 자체가 병렬, 줄만 길어짐 |
| 기본 풀 | 40 (고정) | 조절 불가, 모든 def가 공유 |

In [13]:
import os
print(f"이 컴퓨터의 CPU 코어 수: {os.cpu_count()}")
print(f"→ CPU 추론이라면 max_workers={os.cpu_count()} 정도가 적당합니다.")

이 컴퓨터의 CPU 코어 수: 12
→ CPU 추론이라면 max_workers=12 정도가 적당합니다.


### §4.6 패턴 선택 — 일반 def vs 전용 풀

| | 일반 `def` | **전용 풀 + run_in_executor** |
|---|-----------|------------------------------|
| 코드 | 가장 단순 | 한 줄 더 |
| 크기 제어 | ❌ 불가(40 고정) | ✅ 직접 결정 |
| 부하 격리 | ❌ 모두 공유 | ✅ 추론 전용 |
| 의도 표현 | 암묵적 | ✅ 명시적 |

> 이 과정의 표준은 **전용 풀 + run_in_executor** 입니다.

### ✅ §4 체크포인트
1. 스레드풀이란 무엇이며, 왜 '미리 정해진 수'로 만듭니까?
2. `run_in_executor`는 무슨 일을 합니까? `await`의 역할은?
3. CPU 추론과 GPU 추론의 권장 풀 크기는 각각 얼마입니까?
4. 일반 def도 동작하는데 굳이 전용 풀을 쓰는 이유 두 가지는?

## §5. 에러 핸들링

### §5.1 검증을 통과한 입력도 서버를 터뜨린다

Day 2의 입력 검증은 잘못된 입력을 **문 앞에서** 막았습니다.
그런데 입력이 멀쩡해도 서버 안에서는 언제든 터질 수 있습니다.
(모델 파일 손상, GPU 메모리 부족, 이미지 디코딩 실패 …)

대비가 없으면:
- **보안** — 내부 오류 내용이 사용자에게 노출 → 공격 단서
- **운영** — 기록이 없으면 "어제 왜 죽었지?"에 답할 단서 0
- **최악** — 처리 안 된 오류가 서버 프로세스를 내림

> 목표 세 줄: 서버는 살아있고, 기록엔 상세를 남기고, 사용자에겐 안전한 메시지만.

### §5.2 전역 예외 처리 — 한곳의 그물

엔드포인트마다 `try/except`를 다는 대신, 맨 아래에 **그물 하나**를 칩니다.

```
오류 → 전역 핸들러 → ┬─ 기록(로그): 상세 전부
                     └─ 사용자: 500 + "서버 내부 오류" 한 줄
```

### §5.3 로깅 — `print`를 졸업할 시간

| `print()` | `logging` |
|-----------|-----------|
| 시각 없음 | 시각·모듈명 자동 |
| 심각도 구분 없음 | INFO→WARNING→ERROR→CRITICAL |
| 끄거나 보낼 수 없음 | 화면·파일·외부로 전송 가능 |

In [14]:
%%writefile app/logger.py
"""로그 형식 정의 — 다른 프로젝트에 그대로 들고 가도 되는 범용 부품"""
import logging
import sys

def setup_logger(name="ml_api", level=logging.INFO):
    logger = logging.getLogger(name)
    logger.setLevel(level)
    if logger.handlers:           # 중복 등록 방지
        return logger
    handler = logging.StreamHandler(sys.stdout)
    fmt = logging.Formatter(
        "%(asctime)s %(levelname)s [%(name)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    handler.setFormatter(fmt)
    logger.addHandler(handler)
    return logger

Overwriting app/logger.py


### §5.4 미들웨어 — 모든 요청이 지나는 단 하나의 길목

톨게이트처럼 들어올 때 시계를 켜고, 나갈 때 한 줄을 남깁니다.

```
POST /predict -> 200 (0.045s)
```
한 번만 등록하면 이후 모든 엔드포인트에 적용됩니다.

In [15]:
%%writefile app/error_handlers.py
"""전역 예외 처리 + 요청/응답 로깅 미들웨어 — 범용 부품"""
import time
from fastapi import Request
from fastapi.responses import JSONResponse
from app.logger import setup_logger

logger = setup_logger("ml_api")

def register_error_handling(app):
    """앱에 전역 예외 핸들러와 로깅 미들웨어를 등록합니다."""

    # 1) 요청/응답 자동 기록 미들웨어
    @app.middleware("http")
    async def log_requests(request: Request, call_next):
        t0 = time.time()
        response = await call_next(request)
        dt = time.time() - t0
        level = logger.warning if response.status_code >= 400 else logger.info
        level(f"{request.method} {request.url.path} "
              f"-> {response.status_code} ({dt:.3f}s)")
        return response

    # 2) 전역 예외 처리 (못 잡은 오류가 모이는 그물)
    @app.exception_handler(Exception)
    async def global_exception_handler(request: Request, exc: Exception):
        # 기록엔 상세 전부 (추적 가능)
        logger.error(f"Unhandled error on {request.url.path}: "
                     f"{type(exc).__name__}: {exc}", exc_info=True)
        # 사용자에겐 안전한 한 줄만
        return JSONResponse(
            status_code=500,
            content={"detail": "서버 내부 오류가 발생했습니다."},
        )

Overwriting app/error_handlers.py


### ✅ §5 체크포인트
1. 입력 검증(Day 2)을 통과한 요청도 서버를 터뜨릴 수 있는 예를 드세요.
2. 핸들러마다 try/except를 다는 대신 전역 핸들러를 쓰는 이유는?
3. `print` 대신 `logging`을 쓰면 좋은 점 세 가지는?
4. 미들웨어가 '톨게이트'에 비유되는 이유는?

## §6. 최종 조립 & 검증

### §6.1 오늘 만든 부품 셋 + 어제까지의 부품

```
오늘의 신상 부품:
  ├─ logger.py          (로그 형식)
  ├─ error_handlers.py  (전역 예외 처리 + 미들웨어)
  └─ (전용 풀 + run_in_executor)
지휘부: main_final.py
→ Day 1·2 부품(model_utils.py, schemas.py)은 한 줄도 안 고침!
```

In [24]:
%%writefile app/main_final.py
"""Day 3 최종 서버 — 비동기 추론 + 전역 예외 처리 + 로깅 미들웨어"""
import asyncio
import torch
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI, HTTPException

from app.model_utils import load_model, predict
from app.schemas import PredictRequest, PredictResponse, HealthResponse
from app.error_handlers import register_error_handling
from app.logger import setup_logger

logger = setup_logger("ml_api")
app = FastAPI(title="MNIST API (Day 3 Final)", version="3.0.0")

# 전역 예외 처리 + 요청 로깅 미들웨어 등록 (한 줄)
register_error_handling(app)

# 추론 전용 스레드풀 (CPU 추론 → 코어 수에 맞춤)
import os
pool = ThreadPoolExecutor(max_workers=os.cpu_count() or 4)

# 모델은 서버 시작 시 한 번만 로드
try:
    model = load_model("models/mnist_state_dict.pth")
    model_loaded = True
    logger.info("모델 로드 완료")
except Exception as e:
    model = None
    model_loaded = False
    logger.error(f"모델 로드 실패: {e}")


@app.get("/health", response_model=HealthResponse)
async def health():
    return HealthResponse(status="healthy", model_loaded=model_loaded)


def _run_inference(pixel_values):
    """옆방(스레드풀)에서 실행될 무거운 추론."""
    tensor = torch.tensor(pixel_values, dtype=torch.float32).reshape(1, 1, 28, 28)
    return predict(model, tensor)


@app.post("/predict", response_model=PredictResponse)
async def predict_digit(request: PredictRequest):
    if not model_loaded:
        raise HTTPException(status_code=503, detail="모델이 로드되지 않았습니다.")
    loop = asyncio.get_event_loop()
    # 무거운 추론만 옆방으로 위임 → 루프는 자유
    result = await loop.run_in_executor(pool, _run_inference, request.pixel_values)
    return PredictResponse(
        label=result["label"],
        confidence=result["confidence"],
        probabilities=result["probabilities"] if request.return_probabilities else None,
    )

Overwriting app/main_final.py


### §6.2 검증 — 직접 두드려 본다

실제 MNIST 추론으로 동시 요청을 늘려 가며 **속도와 생존** 둘 다 확인합니다.

In [25]:
# 최종 서버 실행
serve_in_thread("app.main_final:app", port=8000)
time.sleep(2)

2026-06-11 12:50:20 INFO [ml_api] 모델 로드 완료
서버 실행됨: http://127.0.0.1:8000


**STEP 1 — 헬스체크** (모델이 올라왔는지)

In [26]:
r = requests.get("http://127.0.0.1:8000/health")
print(f"상태 코드: {r.status_code}")
print(f"응답: {r.json()}")

2026-06-11 12:50:26 INFO [ml_api] GET /health -> 200 (0.000s)
상태 코드: 200
응답: {'status': 'healthy', 'model_loaded': True}


**STEP 2 — 실제 MNIST 이미지 준비**

In [27]:
from torchvision import datasets, transforms

test_dataset = datasets.MNIST(
    root="data", train=False, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
)
test_image, true_label = test_dataset[0]
pixel_values = test_image.flatten().tolist()
print(f"정답 레이블: {true_label}, 픽셀 수: {len(pixel_values)}")

정답 레이블: 7, 픽셀 수: 784


In [28]:
# 단일 추론
r = requests.post("http://127.0.0.1:8000/predict",
                  json={"pixel_values": pixel_values, "return_probabilities": False})
print(f"상태 코드: {r.status_code}")
print(f"응답: {r.json()}")

2026-06-11 12:50:40 INFO [ml_api] POST /predict -> 200 (0.004s)
상태 코드: 200
응답: {'label': 7, 'confidence': 1.0, 'probabilities': None, 'model_version': '1.0.0'}


**STEP 3 — 동시 요청 확대 (1 → 2 → 4 → 8개)**
개선 서버는 요청 수가 늘어도 전체 시간이 거의 안 늘어야 합니다.

In [29]:
import threading, time

def fire_predict(n, pixels):
    times = {}
    def worker(i):
        t0 = time.time()
        requests.post("http://127.0.0.1:8000/predict",
                      json={"pixel_values": pixels})
        times[i] = time.time() - t0
    threads = [threading.Thread(target=worker, args=(i,)) for i in range(n)]
    t0 = time.time()
    for t in threads: t.start()
    for t in threads: t.join()
    return time.time() - t0

for n in [1, 2, 4, 8]:
    total = fire_predict(n, pixel_values)
    print(f"  동시 {n}개 요청 → 전체 {total:.2f}초")

2026-06-11 12:50:44 INFO [ml_api] POST /predict -> 200 (0.004s)
  동시 1개 요청 → 전체 0.01초
2026-06-11 12:50:44 INFO [ml_api] POST /predict -> 200 (0.006s)
2026-06-11 12:50:44 INFO [ml_api] POST /predict -> 200 (0.008s)
  동시 2개 요청 → 전체 0.02초
2026-06-11 12:50:44 INFO [ml_api] POST /predict -> 200 (0.008s)
2026-06-11 12:50:44 INFO [ml_api] POST /predict -> 200 (0.010s)
2026-06-11 12:50:44 INFO [ml_api] POST /predict -> 200 (0.012s)
2026-06-11 12:50:44 INFO [ml_api] POST /predict -> 200 (0.013s)
  동시 4개 요청 → 전체 0.03초
2026-06-11 12:50:45 INFO [ml_api] POST /predict -> 200 (0.013s)
2026-06-11 12:50:45 INFO [ml_api] POST /predict -> 200 (0.017s)
2026-06-11 12:50:45 INFO [ml_api] POST /predict -> 200 (0.018s)
2026-06-11 12:50:45 INFO [ml_api] POST /predict -> 200 (0.020s)
2026-06-11 12:50:45 INFO [ml_api] POST /predict -> 200 (0.024s)
2026-06-11 12:50:45 INFO [ml_api] POST /predict -> 200 (0.024s)
2026-06-11 12:50:45 INFO [ml_api] POST /predict -> 200 (0.025s)
2026-06-11 12:50:45 INFO [ml_api] POST

**STEP 4 — 추론 중 맥박** (`/health` 0.0초 확인)

In [30]:
threading.Thread(target=lambda: [
    requests.post("http://127.0.0.1:8000/predict", json={"pixel_values": pixel_values})
    for _ in range(4)
]).start()
time.sleep(0.3)
t0 = time.time()
r = requests.get("http://127.0.0.1:8000/health")
print(f"추론 도는 중 /health 응답: {time.time()-t0:.3f}초 (status={r.status_code})")

2026-06-11 12:50:50 INFO [ml_api] POST /predict -> 200 (0.002s)
2026-06-11 12:50:50 INFO [ml_api] POST /predict -> 200 (0.003s)
2026-06-11 12:50:50 INFO [ml_api] POST /predict -> 200 (0.003s)
2026-06-11 12:50:50 INFO [ml_api] POST /predict -> 200 (0.005s)
2026-06-11 12:50:51 INFO [ml_api] GET /health -> 200 (0.000s)
추론 도는 중 /health 응답: 0.004초 (status=200)


**STEP 5 — 비정상 입력 + 생존 확인**
크기 틀린 픽셀(422)을 보내도 서버는 살아있고, 직후 정상 추론이 됩니다.

In [31]:
# 비정상: 픽셀 100개만 전송 → 422
r = requests.post("http://127.0.0.1:8000/predict", json={"pixel_values": [0.0]*100})
print(f"비정상 입력 상태 코드: {r.status_code} (422 기대)")

# 직후 정상 추론이 되는지 (생존 확인)
r = requests.post("http://127.0.0.1:8000/predict", json={"pixel_values": pixel_values})
print(f"직후 정상 추론 상태 코드: {r.status_code} (200 기대)")
print("→ 서버는 비정상 요청에도 죽지 않았습니다. 로그에 WARNING ... -> 422 흔적이 남습니다.")

2026-06-11 12:50:54 WARNING [ml_api] POST /predict -> 422 (0.001s)
비정상 입력 상태 코드: 422 (422 기대)
2026-06-11 12:50:54 INFO [ml_api] POST /predict -> 200 (0.002s)
직후 정상 추론 상태 코드: 200 (200 기대)
→ 서버는 비정상 요청에도 죽지 않았습니다. 로그에 WARNING ... -> 422 흔적이 남습니다.


### §6.3 정리 & 다음 — Day 4 예고

오늘 한 줄 서던 서버가, 동시 요청에 견디는 서버가 되었습니다.

| 배운 것 | 핵심 |
|---------|------|
| 동기 vs 비동기 | 9초와 3초를 가른 건 '틈을 쓰는 전략' |
| 이벤트 루프 | 일꾼이 한 명이라, 붙들리면 헬스체크까지 멈춘다 |
| `run_in_executor` | 무거운 추론만 옆방(전용 풀)으로 (CPU≈코어수, GPU 1~2) |
| 전역 예외 처리 | 기록엔 상세, 응답엔 안전한 한 줄, 서버는 생존 |
| logging·미들웨어 | 모든 호출을 `메서드 경로 -> 상태 (시간)`으로 자동 기록 |

**다음 — Day 4:** Streamlit으로 웹 UI (코드 없이 클릭으로 쓰는 모델)
이미지 업로드 → 추론 → 결과 표시. Day 1의 '클라이언트–서버 분리' 그림의 완성!

### 정리 — 서버 종료
실습이 끝나면 백그라운드 서버를 정리합니다.

In [32]:
stop_server(8000)
print("서버 종료 완료")

서버 종료 완료
